**Self RAG Implementation**

Some Important Specifications -
This notebook implements a basic Naive RAG (Retrieval-Augmented Generation) pipeline using:

1. Supabase as vector database
2. PyMuPDF for PDF reading
3. Sentence Transformers embeddings
4. BAAI/bge-small-en-v1.5 embedding model
5. vecs as pgvector wrapper
6. Groq for LLM inference

This colab notebook is a breif introduction to how a normal RAG model works for starters. A production-grade RAG is different in some aspects while the logic remains the same.

A Normal RAG deals with the problem of
Bad Retrival
Hallucination
Missing Evidence
A Normal RAG works by fetching the relevant documents for a query but a self RAG improves this relevance by first -
1. Retrival Decision: checking if the retrival should happen for a query.
2. Retrival Grading: Next checking if documents are relevant or not
3. Faithfulness/Grounded Verification: if the generated response is grounded or not
4. Completeness Checking: Does the response actually answer the user's query. It check for unsupported claims, contradictions, fabricated facts, speculative language. Since it evaluated if an answer is correct based on user query it is hard to evaluate it hence only an LLM can be used as evaluator.


**NOTE:** All these checks are done in iterative feedback loops

**Model Workflow**

User Query
   ↓
Retrieval Decision Classifier
   ↓
Vector Search
   ↓
Cross Encoder Re-ranker
   ↓
Top Documents
   ↓
LLM Generation
   ↓
Groundedness Check
   ↓
Hallucination Check
   ↓
Retry or Final Answer

#Step 1: Install Libraries

In [ ]:

!pip install -q \
    langchain \
    langchain-community \
    langchain-text-splitters \
    sentence-transformers \
    transformers \
    supabase \
    openai \
    pypdf \
    tiktoken

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_KEY"

SUPABASE_URL = "YOUR_SUPABASE_URL"
SUPABASE_KEY = "YOUR_SUPABASE_SERVICE_ROLE_KEY"

#Step 2: Supabase Setup

In [ ]:
# ============================================================
# SUPABASE SQL
# RUN THIS INSIDE SUPABASE SQL EDITOR
# ============================================================

"""
create extension if not exists vector;

create table documents (
  id bigserial primary key,
  content text,
  metadata jsonb,
  embedding vector(384)
);

create or replace function match_documents (
  query_embedding vector(384),
  match_threshold float,
  match_count int
)
returns table (
  id bigint,
  content text,
  metadata jsonb,
  similarity float
)
language sql stable
as $$
  select
    id,
    content,
    metadata,
    1 - (documents.embedding <=> query_embedding) as similarity
  from documents
  where 1 - (documents.embedding <=> query_embedding) > match_threshold
  order by similarity desc
  limit match_count;
$$;
"""

#Step 3: PDF Extraction



In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import glob
from langchain_community.document_loaders import PyPDFLoader

pdf_folder = "/content/drive/MyDrive/rag_docs/*.pdf"

documents = []

for file in glob.glob(pdf_folder):

    loader = PyPDFLoader(file)

    documents.extend(loader.load())

print(f"Loaded {len(documents)} pages")


#Step 4: Chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = splitter.split_documents(documents)

print(f"Total Chunks: {len(chunks)}")

#Step 5: Load Embedding Model

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

#Step 6: Embedding Storage to Supabase

In [ ]:
from supabase import create_client

supabase = create_client(
    SUPABASE_URL,
    SUPABASE_KEY
)

In [ ]:
for chunk in chunks:

    vector = embedding_model.encode(
        chunk.page_content
    ).tolist()

    supabase.table("documents").insert({
        "content": chunk.page_content,
        "metadata": {
            "source": chunk.metadata.get("source")
        },
        "embedding": vector
    }).execute()

print("Upload Complete")


#Step 7: Retrival Decision Classifier

In [ ]:
from transformers import pipeline

retrieval_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

def needs_retrieval(query):

    labels = [
        "needs retrieval",
        "no retrieval"
    ]

    result = retrieval_classifier(
        query,
        labels
    )

    top_label = result["labels"][0]

    return top_label == "needs retrieval"

#Step 8: Document Relevance (Re-Ranker)

In [ ]:
def vector_search(query, k=8):

    query_embedding = embedding_model.encode(
        query
    ).tolist()

    result = supabase.rpc(
        "match_documents",
        {
            "query_embedding": query_embedding,
            "match_threshold": 0.4,
            "match_count": k
        }
    ).execute()

    return result.data

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

def rerank(query, docs, top_k=4):

    pairs = [
        [query, d["content"]]
        for d in docs
    ]

    scores = reranker.predict(pairs)

    rescored = []

    for doc, score in zip(docs, scores):

        rescored.append((doc, score))

    rescored = sorted(
        rescored,
        key=lambda x: x[1],
        reverse=True
    )

    return [x[0] for x in rescored[:top_k]]

#Step 9: Answer Generation

In [ ]:
from openai import OpenAI

client = OpenAI()

In [ ]:
def generate_answer(query, docs):

    context = "\n\n".join(
        [d["content"] for d in docs]
    )

    prompt = f"""
You are a grounded AI assistant.

Answer ONLY using the provided context.

If answer is not present in context,
say you do not know.

Question:
{query}

Context:
{context}

Answer:
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content

#Step 10: Groundness Check

In [ ]:
def groundedness_check(query, docs, answer):

    context = "\n\n".join(
        [d["content"] for d in docs]
    )

    prompt = f"""
You are a grounding evaluator.

Question:
{query}

Documents:
{context}

Answer:
{answer}

Determine whether ALL claims
in the answer are supported
by the documents.

Output ONLY:
SUPPORTED
or
UNSUPPORTED
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    result = response.choices[0].message.content.strip()

    return result

#Step 11: Completeness Check

In [ ]:
def hallucination_check(query, docs, answer):

    context = "\n\n".join(
        [d["content"] for d in docs]
    )

    prompt = f"""
You are a hallucination detector.

Question:
{query}

Documents:
{context}

Answer:
{answer}

Check whether the answer contains:
- fabricated facts
- unsupported claims
- contradictions

Output ONLY:
YES
or
NO
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    result = response.choices[0].message.content.strip()

    return result

#Step 12: Query Re-Write on Feedback

In [ ]:
def rewrite_query(query):

    prompt = f"""
Rewrite the query for better retrieval.

Original Query:
{query}

Improved Query:
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()

#Step 13: Self RAG Pipeline

In [ ]:
def self_rag(query):

    print("=" * 60)
    print(f"QUERY: {query}")
    print("=" * 60)

    # --------------------------------------------------------
    # STEP 1: RETRIEVAL DECISION
    # --------------------------------------------------------

    retrieve = needs_retrieval(query)

    print(f"\nNeeds Retrieval: {retrieve}")

    # --------------------------------------------------------
    # NO RETRIEVAL PATH
    # --------------------------------------------------------

    if not retrieve:

        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {
                    "role": "user",
                    "content": query
                }
            ]
        )

        return response.choices[0].message.content

    # --------------------------------------------------------
    # STEP 2: VECTOR SEARCH
    # --------------------------------------------------------

    docs = vector_search(query)

    print(f"\nRetrieved Docs: {len(docs)}")

    # --------------------------------------------------------
    # STEP 3: RERANK
    # --------------------------------------------------------

    docs = rerank(query, docs)

    print(f"Reranked Docs: {len(docs)}")

    # --------------------------------------------------------
    # STEP 4: GENERATE ANSWER
    # --------------------------------------------------------

    answer = generate_answer(query, docs)

    print("\nGenerated Answer:\n")
    print(answer)

    # --------------------------------------------------------
    # STEP 5: GROUNDEDNESS CHECK
    # --------------------------------------------------------

    grounded = groundedness_check(
        query,
        docs,
        answer
    )

    print(f"\nGroundedness: {grounded}")

    # --------------------------------------------------------
    # STEP 6: HALLUCINATION CHECK
    # --------------------------------------------------------

    hallucinated = hallucination_check(
        query,
        docs,
        answer
    )

    print(f"Hallucination: {hallucinated}")

    # --------------------------------------------------------
    # STEP 7: FINAL DECISION
    # --------------------------------------------------------

    if grounded == "SUPPORTED" and hallucinated == "NO":

        print("\nFINAL STATUS: PASSED")

        return answer

    # --------------------------------------------------------
    # RETRY LOOP
    # --------------------------------------------------------

    print("\nRetrying with rewritten query...")

    improved_query = rewrite_query(query)

    print(f"\nImproved Query:\n{improved_query}")

    docs = vector_search(improved_query)

    docs = rerank(improved_query, docs)

    answer = generate_answer(
        improved_query,
        docs
    )

    return answer



# ============================================================
# TEST QUERY
# ============================================================

response = self_rag(
    "Explain Redis persistence mechanisms"
)

print("\n")
print("=" * 60)
print("FINAL RESPONSE")
print("=" * 60)

print(response)